In [ ]:
from notebookutils import mssparkutils
from datetime import datetime

try:
    batch_id = mssparkutils.widgets.get("batch_id")
except Exception:
    batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

print(f"batch_id = {batch_id}")

In [19]:
from pyspark.sql import functions as F 

# 1. Silver transform
BRONZE = "review_bronze"
SILVER = "review_silver"

df_review = spark.table(BRONZE)

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, 21, Finished, Available, Finished, False)

In [20]:
base_cols = ["review_id", "business_id", "user_id", "stars", "useful", "funny", "cool", "text","date",
    "_ingest_ts", "_source_file", "_batch_id"
]

df_review = df_review.select(*base_cols)

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, 22, Finished, Available, Finished, False)

In [21]:
df_review_silver = (df_review
    .withColumn("stars", F.col("stars").cast("double"))

    .withColumn("useful",F.coalesce(F.col("useful"), F.lit(0)).cast("long"))
    .withColumn("funny", F.coalesce(F.col("funny"), F.lit(0)).cast("long"))
    .withColumn("cool", F.coalesce(F.col("cool"), F.lit(0)).cast("long"))

    .withColumn("text", F.trim(F.col("text")))

    .filter(F.col("review_id").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("business_id").isNotNull())
    .filter(F.col("stars").isNotNull())
    .filter((F.col("stars") >= 1) & (F.col("stars") <= 5))

    .dropDuplicates(["review_id"])
)


StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, 23, Finished, Available, Finished, False)

In [22]:
(df_review_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)
)

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, 24, Finished, Available, Finished, False)

In [23]:
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, 25, Finished, Available, Finished, False)

ExitValue: SUCCESS

In [ ]:
#2. Data Quality Check

df_review_silver_DQ = spark.table(SILVER)
print("review_silver rows = ", df_review_silver_DQ.count())

dup = df_review_silver_DQ.groupBy("review_id").count().filter("count > 1").count()

print("duplicate review_id=",dup)

df_review_silver_DQ.select(
    F.sum(F.col("review_id").isNull().cast("int")).alias("null_review_id"),
    F.sum(F.col("business_id").isNull().cast("int")).alias("null_business_id"),
    F.sum(F.col("user_id").isNull().cast("int")).alias("null_user_id"),
    F.sum(F.col("stars").isNull().cast("int")).alias("null_stars"),
    F.sum(F.col("date").isNull().cast("int")).alias("null_date"),
).show()

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, -1, Cancelled, , Cancelled, True)

In [ ]:
invalid_stars = df_review_silver_DQ.filter((F.col("stars") < 1) | (F.col("stars") > 5)).count()
print("invalid stars =", invalid_stars)

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, -1, Cancelled, , Cancelled, True)

In [ ]:
negative_votes = df_review_silver_DQ.filter(
    (F.col("useful") < 0) | (F.col("funny") < 0) | (F.col("cool") < 0)
).count()
print("negative vote rows =", negative_votes)

df_review_silver_DQ.filter(
    (F.col("useful") < 0) |
    (F.col("funny") < 0) |
    (F.col("cool") < 0)
).show(truncate=False)

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, -1, Cancelled, , Cancelled, True)

In [ ]:
# 3. DQ remediation
df_review_silver = (
    df_review_silver
    .withColumn(
        "useful",
        F.when(F.col("useful") < 0, 0)
        .otherwise(F.coalesce(F.col("useful"), F.lit(0)))
        .cast("int")
    )
    .withColumn(
        "funny",
        F.when(F.col("funny") < 0, 0)
        .otherwise(F.coalesce(F.col("funny"), F.lit(0)))
        .cast("int")
    )
    .withColumn(
    "cool",
    F.when(F.col("cool") < 0, 0)
    .otherwise(F.coalesce(F.col("cool"), F.lit(0)))
    .cast("int")
    )
)

(
    df_review_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("review_silver")
)

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, -1, Cancelled, , Cancelled, True)

In [ ]:
df_fixed = spark.table("review_silver")

negative_votes_after = df_fixed.filter(
    (F.col("useful") < 0) |
    (F.col("funny") < 0) |
    (F.col("cool") < 0)
).count()

print("negative vote rows after fix =", negative_votes_after)

StatementMeta(, 14dab947-dd02-45f5-ad7c-68426eb927be, -1, Cancelled, , Cancelled, True)